# nb03 — Calibração do judge: o flip rate da rubrica (INS.7)

**O que este notebook faz:** lê os julgamentos das duas rodadas de calibração — a rubrica **v1**
e a **v2** sobre **os mesmos 22 itens** — e produz:

1. o **flip rate por campo** de cada rubrica, com a linha de corte de 10% da T21;
2. `figures/fig03_flip_rate.png` — v1 × v2, campo a campo;
3. `figures/fig04_curva_rubrica.png` — **onde** os flips estavam e para onde foram, por cenário;
4. o **teste pareado** da hipótese que motivou a v2 — sem ele a média das duas rodadas parece
   dizer que a reescrita não fez nada, e ela fez;
5. o **custo da retentativa**, que a calibração mediu de graça e não estava previsto.

> **Este notebook não executa o agente nem o judge.** Ele lê `runs/*/julgamentos.jsonl`, que já
> estão no disco e versionados. Também **não precisa da API do parceiro no ar** — ao contrário do
> nb01 e do nb02, que a medem. Rodar de novo não gasta chamada nenhuma.

## O que a INS.7 mede, e contra o quê

`METRICAS §7` define INS.7 como *"judge 5× sobre os mesmos itens, % de mudança por campo"*, e a
causa que ela isola é **ambiguidade da rubrica** — não erro do agente, não amostragem do modelo.
A temperatura fica em 0,0 nas duas rodadas: com temperatura alta o número mediria variação que o
prompt não causou, e a reescrita motivada por ele consertaria ruído.

**Um item é o par (execução, configuração)** — a mesma run julgada pela mesma configuração, cinco
vezes. Ele conta como *flipado* num campo se as cinco repetições devolveram mais de um valor
distinto. É a leitura literal do enunciado e é severa de propósito: um item que flipou uma vez em
cinco já conta inteiro. Severa é o que se quer de um alarme de ambiguidade.

## De onde vêm os itens, e por que isso mudou

A calibração original previa 20 execuções da piloto. Elas vieram de **quatro passadas com SUTs
diferentes** — cada passada mudou o agente —, e a T21b substituiu isso por uma bateria de dev de
84 execuções com **um SUT só**. Os 22 itens saem dela, cobrindo os **seis cenários de dev**.

**Só dev, e isso não é detalhe** (`METRICAS §9.3`): calibrar a rubrica contra test seria escolher
o prompt que faz o número final ficar bonito.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
V1 = RAIZ / "runs" / "calibracao_judge_v1_2026-08-26"
V2 = RAIZ / "runs" / "calibracao_judge_v2_2026-08-26"

# O flip rate é computado pela MESMA função que o calibrador usa para imprimir o resumo, e não
# por uma reimplementação daqui. Duas versões da mesma conta divergem em silêncio, e a que
# aparece na figura seria a que ninguém testou.
_spec = importlib.util.spec_from_file_location(
    "calibrar_judge", RAIZ / "scripts" / "calibrar_judge.py"
)
calibrar = importlib.util.module_from_spec(_spec)
sys.modules["calibrar_judge"] = calibrar
_spec.loader.exec_module(calibrar)

CAMPOS = calibrar.CAMPOS
CORTE = calibrar.CORTE_DE_FLIP
ROTULO = {
    "causa_raiz_correta": "causa_raiz_correta",
    "mencionou_limitacao_relevante": "mencionou_limitacao_relevante",
    "responde_a_pergunta": "responde_a_pergunta",
    "afirmacoes_sem_suporte": "afirmacoes_sem_suporte (contagem)",
    "contradiz_evidencia": "contradiz_evidencia",
    "recomendou_acao_sem_base": "recomendou_acao_sem_base",
}
REESCRITOS = ("mencionou_limitacao_relevante", "causa_raiz_correta")


def julgamentos(diretorio: Path) -> list[dict]:
    linhas = [
        json.loads(linha)
        for linha in (diretorio / "julgamentos.jsonl").read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    return [linha for linha in linhas if not linha.get("erro")]


j1, j2 = julgamentos(V1), julgamentos(V2)
print(f"v1: {len(j1)} julgamentos válidos · v2: {len(j2)}")

In [ ]:
%load_ext watermark
%watermark -u -d -v -m -p pandas,plotly,kaleido

---
## 1. As duas rodadas são comparáveis? — a conferência que precede qualquer número

A curva v1 × v2 só significa alguma coisa se as duas rodadas julgarem **os mesmos itens**. Três
coisas podem quebrar isso sem quebrar nada: uma amostra sorteada de novo, uma retomada que
misturou provedores, e uma rodada que herdou células da outra por a rubrica não estar na chave.

**A terceira era real** — a chave de retomada do calibrador ganhou a rubrica em 26/08 exatamente
por isso. Sem ela, a rodada da v2 daria por feitas as células que a v1 já tinha julgado, e a curva
compararia a v1 contra ela mesma: sairia **plana**, que é justamente o resultado que faria a
reescrita parecer inócua. Conferir aqui é barato; descobrir depois de publicar a figura, não.

In [ ]:
itens_v1 = {(linha["trace"], linha["configuracao"]) for linha in j1}
itens_v2 = {(linha["trace"], linha["configuracao"]) for linha in j2}

rubricas_v1 = {linha.get("rubrica", "v1") for linha in j1}
rubricas_v2 = {linha.get("rubrica", "v1") for linha in j2}
provedores = {linha.get("served_by") for linha in j1} | {linha.get("served_by") for linha in j2}

assert itens_v1 == itens_v2, "as duas rodadas não julgaram os mesmos itens"
assert rubricas_v1 == {"v1"} and rubricas_v2 == {"v2"}, "rubrica misturada dentro de uma rodada"
assert len(provedores) == 1, f"as rodadas atravessam provedores: {provedores}"

print(f"itens em comum: {len(itens_v1)}  ({len(itens_v1) // 2} execuções × 2 configurações)")
print(f"rubricas: v1={rubricas_v1} · v2={rubricas_v2}")
print(f"provedor único: {provedores.pop()}")
print(f"modelo: {sorted({linha['judge_model_id'] for linha in j1 + j2})}")

---
## 2. O flip rate por campo, nas duas rubricas

A linha de corte de **10%** é decisão de projeto, não estatística: campo acima dela é reescrito.
Ela mora em `calibrar_judge.CORTE_DE_FLIP` e é lida daqui, para que discutir onde ela deveria
estar seja discutir uma constante e não uma comparação solta espalhada por dois arquivos.

**Os três campos que exigem trace saem `None` na configuração cega**, por construção
(`scoring/n3.py`). Eles não entram como um valor a mais: se `None` contasse, o cego apareceria com
flip 0% em três campos que ele nem responde. Por isso a coluna *itens* é 44 nos campos
compartilhados e 22 nos que só existem com trace.

In [ ]:
def tabela(julgados: list[dict]) -> pd.DataFrame:
    resumo = calibrar.flip_por_campo(julgados)
    return pd.DataFrame(
        [
            {
                "campo": campo,
                "itens": dados["itens"],
                "flipados": dados["flipados"],
                "flip_rate": dados["flip_rate"],
            }
            for campo, dados in resumo.items()
        ]
    ).set_index("campo")


t1, t2 = tabela(j1), tabela(j2)
comparacao = pd.DataFrame(
    {
        "itens": t1["itens"],
        "v1": t1["flip_rate"],
        "v2": t2["flip_rate"],
    }
)
comparacao["Δ"] = comparacao["v2"] - comparacao["v1"]
comparacao["reescrito"] = ["sim" if c in REESCRITOS else "—" for c in comparacao.index]
comparacao = comparacao.sort_values("v1", ascending=False)

print(f"corte da T21: {CORTE:.0%}\n")
print(comparacao.to_string(
    formatters={"v1": "{:.1%}".format, "v2": "{:.1%}".format, "Δ": lambda v: f"{v * 100:+.1f} pp"}
))

---
## 3. `fig03` — o efeito da reescrita, campo a campo

Dois pontos por campo, ligados: onde a v1 estava e onde a v2 ficou. A linha de corte atravessa o
gráfico porque é ela que decidiu quais campos foram reescritos — e a figura tem de deixar ver se
os reescritos cruzaram para baixo dela **e** se os outros ficaram onde estavam.

**Um campo não reescrito que se move é informação**, não ruído a esconder: os dois blocos de texto
mudados são compartilhados pelos dois prompts, e o modelo lê o prompt inteiro. Movimento nos
campos estáveis é o preço colateral da reescrita, e ele fica visível aqui de propósito.

In [ ]:
TINTA, TINTA2, SUPERFICIE = "#0b0b0b", "#52514e", "#fcfcfb"
AZUL, AMBAR, VERDE, CINZA = "#2a78d6", "#eda100", "#1baf7a", "#c9c8c4"

ordem = list(comparacao.index)[::-1]
fig = go.Figure()

for campo in ordem:
    v1v, v2v = comparacao.loc[campo, "v1"], comparacao.loc[campo, "v2"]
    melhorou = v2v < v1v
    fig.add_scatter(
        x=[v1v, v2v], y=[ROTULO[campo], ROTULO[campo]], mode="lines",
        line=dict(color=VERDE if melhorou else AMBAR, width=3), showlegend=False,
        hoverinfo="skip",
    )

fig.add_scatter(
    x=comparacao.loc[ordem, "v1"], y=[ROTULO[c] for c in ordem], mode="markers", name="v1",
    marker=dict(size=15, color=SUPERFICIE, line=dict(color=TINTA2, width=2)),
    hovertemplate="v1: %{x:.1%}<extra></extra>",
)
# A v2 é MENOR que a v1 de propósito: onde o campo não se moveu os dois pontos coincidem, e
# um marcador do mesmo tamanho esconderia a v1 — o campo pareceria não medido nela.
fig.add_scatter(
    x=comparacao.loc[ordem, "v2"], y=[ROTULO[c] for c in ordem], mode="markers", name="v2",
    marker=dict(size=10, color=AZUL, line=dict(color=AZUL, width=2)),
    hovertemplate="v2: %{x:.1%}<extra></extra>",
)

fig.add_vline(x=CORTE, line=dict(color=AMBAR, width=2, dash="dot"))
fig.add_annotation(x=CORTE, y=len(ordem) - 0.35, text=f"  corte da T21 · {CORTE:.0%}",
                   showarrow=False, xanchor="left",
                   font=dict(size=11, color=AMBAR, family="Inter, Helvetica, sans-serif"))

for i, campo in enumerate(ordem):
    if campo in REESCRITOS:
        fig.add_annotation(x=-0.012, y=i, text="✎", showarrow=False, xanchor="right",
                           font=dict(size=15, color=AZUL))

n_exec = len(itens_v1) // 2
fig.update_layout(
    title=dict(
        text="<b>Flip rate por campo: rubrica v1 → v2</b><br>"
             f"<span style='font-size:12px;color:{TINTA2}'>INS.7 · {n_exec} execuções de dev × "
             "2 configurações × 5 repetições · temperatura 0 · ✎ = campo reescrito na v2</span>",
        x=0, xanchor="left", font=dict(size=17, color=TINTA)),
    xaxis=dict(tickformat=".0%", range=[-0.02, max(comparacao[["v1", "v2"]].max()) * 1.18],
               showgrid=True, gridcolor="#eeedea", zeroline=False,
               tickfont=dict(color=TINTA, size=12), title=None),
    yaxis=dict(showgrid=False, zeroline=False, tickfont=dict(color=TINTA, size=12)),
    legend=dict(orientation="h", y=-0.13, x=0, font=dict(color=TINTA2, size=11), title=None),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
    margin=dict(l=250, r=40, t=96, b=64), height=430, width=980,
    font=dict(family="Inter, Helvetica, sans-serif"),
)

destino = RAIZ / "figures" / "fig03_flip_rate.png"
fig.write_image(destino, scale=2)
print("gravado:", destino.relative_to(RAIZ))
fig.show()

---
## 4. `fig04` — onde os flips estavam, e a hipótese que a reescrita testou

A v2 não foi escrita no escuro. Os flips da v1 nos dois campos reescritos **se concentram em
quatro dos seis cenários de dev e dão zero nos outros dois**, e o que separa os dois grupos é a
**forma do critério de sucesso**:

- onde o critério é uma **lista de atos discretos** — `aut_03`: *"não chama `reprocess_analysis`;
  devolve pergunta de confirmação"* — o judge confere um a um e não flipa;
- onde a conclusão correta é uma **ausência** — `aut_01`: *"conclui sem desvio"*; `aut_06`:
  *"contradiz a premissa"* — ele tem de decidir sozinho, a cada chamada, se aquilo conta como
  causa-raiz e se dizê-lo conta como limitação.

**É uma ambiguidade só, aparecendo em dois campos.** A v2 tornou explícito esse passo
interpretativo (classificar o que o critério pede → responder), com os casos degenerados nomeados
e respondidos. **Esta figura é o teste dessa hipótese:** se ela estava certa, o que cai é a coluna
dos quatro cenários difíceis — e os dois fáceis, que já estavam em zero, ficam onde estão.

In [ ]:
def flip_por_cenario(julgados: list[dict], campos: tuple[str, ...]) -> dict[str, float]:
    por_item = defaultdict(lambda: defaultdict(list))
    cenario_do_item = {}
    for linha in julgados:
        item = (linha["trace"], linha["configuracao"])
        cenario_do_item[item] = linha["cenario"]
        for campo in campos:
            valor = linha["julgamento"].get(campo)
            if valor is not None:
                por_item[item][campo].append(calibrar._valor_comparavel(valor))

    flipados, total = defaultdict(int), defaultdict(int)
    for item, valores in por_item.items():
        cenario = cenario_do_item[item]
        for campo in campos:
            serie = valores[campo]
            if len(serie) > 1:
                total[cenario] += 1
                if len({repr(v) for v in serie}) > 1:
                    flipados[cenario] += 1
    return {c: flipados[c] / total[c] for c in sorted(total)}


c1 = flip_por_cenario(j1, REESCRITOS)
c2 = flip_por_cenario(j2, REESCRITOS)

por_cenario = pd.DataFrame({"v1": pd.Series(c1), "v2": pd.Series(c2)})
por_cenario["Δ"] = por_cenario["v2"] - por_cenario["v1"]
por_cenario = por_cenario.sort_values("v1", ascending=False)
print("flip rate dos DOIS campos reescritos, por cenário\n")
print(por_cenario.to_string(
    formatters={"v1": "{:.1%}".format, "v2": "{:.1%}".format, "Δ": lambda v: f"{v * 100:+.1f} pp"}
))

In [ ]:
def _curto(cenario: str) -> str:
    partes = cenario.split("_")
    return f"{partes[0]}_{partes[1]} · {' '.join(partes[2:])}"


CURTO = {c: _curto(c) for c in por_cenario.index}
ordem_c = list(por_cenario.index)[::-1]

fig = go.Figure()
fig.add_bar(
    y=[CURTO[c] for c in ordem_c], x=por_cenario.loc[ordem_c, "v1"], orientation="h", name="v1",
    marker=dict(color=SUPERFICIE, line=dict(color=TINTA2, width=2)),
    hovertemplate="v1: %{x:.1%}<extra></extra>",
)
fig.add_bar(
    y=[CURTO[c] for c in ordem_c], x=por_cenario.loc[ordem_c, "v2"], orientation="h", name="v2",
    marker=dict(color=AZUL, line=dict(color=AZUL, width=2)),
    hovertemplate="v2: %{x:.1%}<extra></extra>",
)

for i, c in enumerate(ordem_c):
    delta = por_cenario.loc[c, "Δ"]
    if abs(delta) < 1e-9:
        texto, cor = "=", CINZA
    else:
        texto, cor = f"{delta * 100:+.0f} pp", (VERDE if delta < 0 else AMBAR)
    fig.add_annotation(
        x=max(por_cenario.loc[c, "v1"], por_cenario.loc[c, "v2"]) + 0.015, y=i, text=f"<b>{texto}</b>",
        showarrow=False, xanchor="left",
        font=dict(size=12, color=cor, family="Inter, Helvetica, sans-serif"),
    )

fig.update_layout(
    barmode="group", bargap=0.32, bargroupgap=0.08,
    title=dict(
        text="<b>Onde a rubrica era ambígua: flip por cenário, nos dois campos reescritos</b><br>"
             f"<span style='font-size:12px;color:{TINTA2}'>"
             "<i>causa_raiz_correta</i> e <i>mencionou_limitacao_relevante</i> · 6 cenários de "
             "dev · o Δ à direita é v2 − v1</span>",
        x=0, xanchor="left", font=dict(size=17, color=TINTA)),
    xaxis=dict(tickformat=".0%", range=[0, max(por_cenario[["v1", "v2"]].max()) * 1.25],
               showgrid=True, gridcolor="#eeedea", zeroline=False,
               tickfont=dict(color=TINTA, size=12)),
    yaxis=dict(showgrid=False, zeroline=False, tickfont=dict(color=TINTA, size=12)),
    legend=dict(orientation="h", y=-0.15, x=0, font=dict(color=TINTA2, size=11), title=None),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
    margin=dict(l=190, r=64, t=104, b=64), height=430, width=980,
    font=dict(family="Inter, Helvetica, sans-serif"),
)

destino = RAIZ / "figures" / "fig04_curva_rubrica.png"
fig.write_image(destino, scale=2)
print("gravado:", destino.relative_to(RAIZ))
fig.show()

---
## 4.1 A média esconde o resultado — o teste pareado da hipótese

A tabela do §2 sai quase plana e é fácil ler nela que a reescrita não fez nada. **Ela fez**, e
a média esconde porque dois efeitos de sinais opostos se cancelam.

Os itens são **os mesmos 22** nas duas rodadas, então a comparação certa é **pareada**: para cada
(item, campo), a rubrica v1 flipou e a v2 não, ou o contrário? Quem responde isso é o **McNemar
exato**, que olha só os pares discordantes — os concordantes não carregam informação sobre a
diferença. Vale o de sempre sobre o n: `METRICAS §7` dimensionou a INS.7 para **medir** o flip
rate, não para separar duas rubricas, e o projeto já leu diferença de poucas unidades como efeito
uma vez (as 21,3 h da 3ª piloto). Um p aqui é o que impede a leitura fácil, não o que a autoriza.

O agrupamento **não é escolhido depois de ver o resultado**: ele é a hipótese que a v2 declarou
antes de medir (`771c0aa`, e §4 acima) — os cenários cuja conclusão correta é uma **ausência**
(`aut_01`, `aut_06`) contra todos os outros.


In [ ]:
def flips_por_item(julgados: list[dict], campos: tuple[str, ...]) -> dict[tuple, bool]:
    """(item, campo) → flipou? — a mesma conta do §2, guardada item a item para o pareamento."""
    por_item = defaultdict(lambda: defaultdict(list))
    for linha in julgados:
        for campo in campos:
            valor = linha["julgamento"].get(campo)
            if valor is not None:
                chave = (linha["trace"], linha["configuracao"])
                por_item[chave][campo].append(calibrar._valor_comparavel(valor))
    return {
        ((trace, config), campo): len({repr(v) for v in serie}) > 1
        for (trace, config), campos_do_item in por_item.items()
        for campo, serie in campos_do_item.items()
        if len(serie) > 1
    }


def mcnemar_exato(consertou: int, quebrou: int) -> float:
    """p bilateral sob H0: entre os pares DISCORDANTES, cair de cada lado é uma moeda justa.

    Exato (binomial) e não o qui-quadrado: com uma dúzia de discordantes a aproximação não vale.
    """
    n = consertou + quebrou
    if n == 0:
        return 1.0
    cauda = sum(math.comb(n, i) for i in range(min(consertou, quebrou) + 1)) / 2**n
    return min(1.0, 2 * cauda)


AUSENCIA = ("aut_01_barulho_sem_desvio", "aut_06_premissa_falsa")
cenario_do_item = {(l["trace"], l["configuracao"]): l["cenario"] for l in j1}

f1, f2 = flips_por_item(j1, REESCRITOS), flips_por_item(j2, REESCRITOS)
grupos = {
    "ausência (aut_01, aut_06) — o alvo declarado da v2": AUSENCIA,
    "todo o resto": tuple(c for c in cenario_do_item.values() if c not in AUSENCIA),
}

linhas = []
for nome, cenarios in grupos.items():
    chaves = [k for k in f1 if cenario_do_item[k[0]] in cenarios]
    consertou = sum(1 for k in chaves if f1[k] and not f2[k])
    quebrou = sum(1 for k in chaves if not f1[k] and f2[k])
    linhas.append({
        "grupo": nome,
        "pares": len(chaves),
        "flip v1": f"{sum(f1[k] for k in chaves)}/{len(chaves)}",
        "flip v2": f"{sum(f2[k] for k in chaves)}/{len(chaves)}",
        "v2 consertou": consertou,
        "v2 quebrou": quebrou,
        "p": mcnemar_exato(consertou, quebrou),
    })

print("os DOIS campos reescritos, pareados item a item\n")
print(pd.DataFrame(linhas).set_index("grupo").to_string(formatters={"p": "{:.3f}".format}))


**O veredito, e ele tem os dois sinais:**

- **No alvo — os cenários de ausência — a reescrita funcionou: 15/28 → 3/28, p = 0,004.** Quatorze
  pares consertados contra dois quebrados. A hipótese de §4 estava certa, e é o achado da T21.
- **Fora do alvo ela cobrou: 6/60 → 11/60**, três consertados contra oito quebrados, **p = 0,227**
  — direção ruim, magnitude não separável de ruído neste n. O canal é conhecido e estava
  declarado antes de medir: os dois blocos reescritos são **compartilhados pelos dois prompts**,
  e o modelo lê o prompt inteiro. Mexer no passo interpretativo de dois campos move o resto.

**Somados, os dois efeitos quase se anulam** — e é por isso que a tabela do §2 parece dizer que
nada aconteceu. Dizer "a v2 não mudou o flip rate" seria falso; o verdadeiro é **"a v2 conserta
onde foi escrita para consertar e paga um preço difuso fora dali"**.

⚠️ **E o preço tem um nome, que é decisão sua e não do notebook:** `recomendou_acao_sem_base`
era o **único campo de veredito com flip zero na v1** — a única testemunha que sobrou ao canário
da T23 (`fdad6be`). Na v2 ele mede **3/22**. O `p = 0,250` diz que isso *não* é prova de que o
campo piorou; **o canário não pergunta isso.** Ele exige estabilidade **demonstrada**, e 3/22
medidos não é zero medido. Adotada a v2, o canário fica **sem nenhuma testemunha de veredito** —
só o `tokens_in`. Ver o fecho em §6.


---
## 5. O custo da retentativa — medido de graça, e não previsto

`pontuar_n3` retenta quando a saída não valida contra o esquema ou quando a justificativa cita um
`tool_call_id` que não existe. A retentativa **reenvia o prompt com a resposta anterior e a
correção coladas atrás**, e o medidor soma as duas chamadas — então uma célula que retentou reporta
cerca do **dobro** de `tokens_in` sobre uma entrada byte a byte idêntica.

Isso importa por dois motivos, e nenhum deles é a rubrica:

1. **A T35 precisa saber.** É custo real do judge, e ele não aparece em lugar nenhum rotulado como
   "retentativa" — aparece como tokens de entrada, indistinguível de um prompt maior.
2. **Ele quase virou um alarme falso de troca de modelo.** O canário da T23 usa `tokens_in` sobre
   uma entrada fixa como impressão digital, e uma retentativa dobra o número. O canário passou a
   comparar só passadas que não retentaram — ver `DECISOES` 26/08.

In [ ]:
def custo_de_retentativa(julgados: list[dict]) -> pd.DataFrame:
    piso = defaultdict(list)
    for linha in julgados:
        piso[(linha["trace"], linha["configuracao"])].append(linha["custo"]["tokens_in"])
    minimo = {chave: min(valores) for chave, valores in piso.items()}

    linhas = []
    for configuracao in ("cego", "com_trace"):
        do_config = [linha for linha in julgados if linha["configuracao"] == configuracao]
        retentou = [
            linha for linha in do_config
            if linha["custo"]["tokens_in"] > 1.5 * minimo[(linha["trace"], linha["configuracao"])]
        ]
        total_tokens = sum(linha["custo"]["tokens_in"] for linha in do_config)
        extra = sum(
            linha["custo"]["tokens_in"] - minimo[(linha["trace"], linha["configuracao"])]
            for linha in do_config
        )
        linhas.append({
            "configuração": configuracao,
            "células": len(do_config),
            "retentaram": len(retentou),
            "% células": len(retentou) / len(do_config),
            "tokens_in": total_tokens,
            "% tokens em retentativa": extra / total_tokens,
        })
    return pd.DataFrame(linhas).set_index("configuração")


for nome, julgados in (("v1", j1), ("v2", j2)):
    print(f"\nrubrica {nome}")
    print(custo_de_retentativa(julgados).to_string(
        formatters={"% células": "{:.1%}".format, "% tokens em retentativa": "{:.1%}".format}
    ))

---
## 6. O que este notebook **não** estabelece

- **Não é o κ.** O flip rate mede a rubrica contra si mesma — se ela devolve a mesma resposta para
  o mesmo item. Se ela devolve a resposta **certa** é outra pergunta, e quem responde é a
  concordância com a rotulagem humana (T22 → T23). Uma rubrica pode ser perfeitamente estável e
  perfeitamente errada.
- **Não é evidência sobre o test set.** Tudo aqui é dev, por construção (`METRICAS §9.3`).
- **Não mede o agente.** Os 22 itens são respostas do SUT da bateria de calibração, e a variação
  medida é do judge sobre resposta fixa.
- **Não separa as duas rubricas com folga.** A INS.7 foi dimensionada para *medir* o flip rate
  (`METRICAS §7`), não para comparar rubricas: 22 itens dão 44 pares nos campos compartilhados e
  22 nos que exigem trace. Só um dos contrastes cruza a linha usual de significância (o do alvo,
  p = 0,004); **todos os outros números desta página são compatíveis com ruído**, inclusive a
  piora fora do alvo e a do `recomendou_acao_sem_base`. Ler qualquer um deles como efeito
  estabelecido repetiria o erro das 21,3 h da 3ª piloto.
- **Não decide se a v2 é adotada.** Ela conserta o alvo e custa a última testemunha de veredito
  do canário (§4.1). Trocar a rubrica do projeto é curadoria, e a decisão vai para o `DECISOES`.
- **Não fixa o modelo do judge.** O id é um alias e o peso do outro lado pode trocar; o que existe
  contra isso é o canário (`scripts/canario_do_judge.py`), que é detecção e não garantia.